In [1]:
from azureml.core import Workspace, Dataset, Datastore

# اتصال به Workspace
subscription_id = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
resource_group = 'forskerpl-n0ybkr-rg'
workspace_name = 'forskerpl-n0ybkr-mlw'

ws = Workspace(subscription_id=subscription_id,
               resource_group=resource_group,
               workspace_name=workspace_name)

# گرفتن datastore
datastore = Datastore.get(ws, "researcher_data")

# خواندن همه فایل‌های parquet در مسیر مشخص
dataset = Dataset.Tabular.from_parquet_files(
    path=[(datastore, 'Zahra/022026/Data/MEDS_MDPS/data/train/*.parquet')]    #     Zahra/Data-07-2025/MDP/MEDS_811/data/train
)

# تبدیل به pandas DataFrame
df = dataset.to_pandas_dataframe()
df.head(15)


Timeout was exceeded in force_flush().
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Resolving access token for scope "https://storage.azure.com/.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


In [2]:
len(df)

448503648

In [3]:
import pandas as pd

# فرض بر این که فایل CSV رو داری
# df = pd.read_csv("your_file.csv")  # یا مستقیم اگر DataFrame آماده‌ست، نیازی نیست

# دسته‌بندی هر subject_id بر اساس وجود کدهای مختلف
m_patients = df[df['code'].str.startswith('M/', na=False)]['subject_id'].unique()
p_patients = df[df['code'].str.startswith('P/', na=False)]['subject_id'].unique()
d_patients = df[df['code'].str.startswith('D/', na=False)]['subject_id'].unique()
s_patients = df[df['code'].str.startswith('S/', na=False)]['subject_id'].unique()

# کل بیماران منحصربه‌فرد
all_patients = df['subject_id'].unique()

# نمایش آمار
print(f"Whole Patients: {len(all_patients)}")
print(f"The patients has M-medication Codes: {len(m_patients)}")
print(f"The patients has D-diagnosis Codes: {len(d_patients)}")
print(f"The patients has P-Procedure Codes: {len(p_patients)}")
print(f"The patients has S-SKS Codes: {len(s_patients)}")


Whole Patients: 1774422
The patients has M-medication Codes: 1208927
The patients has D-diagnosis Codes: 1773917
The patients has P-Procedure Codes: 1701442
The patients has S-SKS Codes: 420586


In [4]:
subject_counts = df['subject_id'].value_counts()


In [5]:
subject_counts

921542     82107
698589     63432
2016077    60873
954669     58832
1964088    57150
           ...  
131767         2
1079388        2
1169509        2
1703144        2
484192         2
Name: subject_id, Length: 1774422, dtype: int64

In [6]:
p_Num = df[df['code'].str.startswith('P/', na=False)]

In [7]:
p_Num

,subject_id,time,code,numeric_value
3,51,2020-02-19 10:57:00,P/UXUD10,NaN
5,51,2020-02-24 08:25:00,P/AAF20,NaN
6,51,2020-02-24 08:25:00,P/ZZ0150,NaN
9,110,2016-05-31 08:08:00,P/UXCD60,NaN
13,110,2016-06-29 11:15:00,P/ZZ0151,NaN
...,...,...,...,...
448503641,2218025,2021-05-12 09:26:00,P/AAF21,NaN
448503642,2218025,2021-05-12 09:26:00,P/KQBA99,NaN
448503643,2218025,2021-05-12 09:26:00,P/ZZ0150,NaN
448503645,2218025,2022-02-17 09:02:00,P/UXRG50,NaN


In [8]:
import pandas as pd

# پیدا کردن سطرهایی که فقط codeهای نوع /P دارن
only_p = df[df['code'].str.startswith('P/', na=False)]

# بیماران با فقط /P کد
subject_ids_only_p = only_p['subject_id'].unique()

# حالا بیماران با codeهای غیر از /P
not_p = df[~df['code'].str.startswith('P/', na=False)]
subject_ids_with_non_p = set(not_p['subject_id'].unique())

# حذف بیمارانی که فقط /P دارن
only_p_ids_to_exclude = [sid for sid in subject_ids_only_p if sid not in subject_ids_with_non_p]

print("Number of patients with only porcedure code: ", only_p_ids_to_exclude)


Number of patients with only porcedure code:  []


In [9]:
df_filtered = df[~df['code'].str.startswith('P/', na=False)]

In [10]:
subject_counts_MDS = df_filtered['subject_id'].value_counts()

In [11]:
subject_counts_MDS

921542     77725
698589     61351
2016077    58264
954669     55553
1964088    52245
           ...  
2044457        2
1047019        2
1331105        2
634058         2
1320663        2
Name: subject_id, Length: 1774422, dtype: int64

In [12]:
subject_counts_df = subject_counts.reset_index()
subject_counts_df.columns = ['subject_id', 'original_count']

subject_counts_MDS_df = subject_counts_MDS.reset_index()
subject_counts_MDS_df.columns = ['subject_id', 'new_count']


In [13]:
import pandas as pd
comparison_df = pd.merge(subject_counts_df, subject_counts_MDS_df, on='subject_id', how='outer')


In [14]:
comparison_df['difference'] =  comparison_df['original_count'] - comparison_df['new_count']


In [15]:
comparison_df = comparison_df.sort_values(by='difference', ascending=False)


In [16]:
comparison_df

,subject_id,original_count,new_count,difference
5,1291159,56181,49847,6334
20,714050,42992,37665,5327
4,1964088,57150,52245,4905
6903,1887674,4651,65,4586
0,921542,82107,77725,4382
...,...,...,...,...
1579607,1221698,15,15,0
1579608,1447820,15,15,0
1579611,911393,15,15,0
1579614,1450366,15,15,0


In [17]:
unchanged_count = (comparison_df['difference'] == 0).sum()
print("unchanged_count", unchanged_count)


unchanged_count 72980


In [18]:
comparison_df['abs_diff'] = comparison_df['difference'].abs()
most_changed = comparison_df.sort_values(by='abs_diff', ascending=False)


In [19]:
print(most_changed.head(10))


      subject_id  original_count  new_count  difference  abs_diff
5        1291159           56181      49847        6334      6334
20        714050           42992      37665        5327      5327
4        1964088           57150      52245        4905      4905
6903     1887674            4651         65        4586      4586
0         921542           82107      77725        4382      4382
11       1169555           49088      45013        4075      4075
33          9512           36660      33002        3658      3658
3         954669           58832      55553        3279      3279
50       2170514           31183      27957        3226      3226
6         350138           52621      49407        3214      3214


In [20]:
changed_df = comparison_df[comparison_df['difference'] != 0]
min_new_count = changed_df['new_count'].min()
lowest_new_count_patients = changed_df[changed_df['new_count'] == min_new_count]


In [21]:
lowest_new_count_patients = lowest_new_count_patients.rename(
    columns={
        'original_count': 'MDPS codes',
        'new_count': 'MDS codes'
    }
)


In [22]:
lowest_new_count_patients

,subject_id,MDPS codes,MDS codes,difference,abs_diff
1286196,356913,31,2,29,29
1437179,1896771,22,2,20,20
1441562,2076088,21,2,19,19
1512270,2003849,18,2,16,16
1505885,1639638,18,2,16,16
...,...,...,...,...,...
1760419,715600,3,2,1,1
1761904,2087106,3,2,1,1
1767528,1328943,3,2,1,1
1773690,426371,3,2,1,1


.str.upper() شرط را case-insensitive می‌کند

بل از فیلتر، ستون را به pd.StringDtype() تبدیل می‌کند؛ این کار رفتار .str را پایدار و قابل پیش‌بینی می‌کند (<NA> به‌جای NaN)



In [23]:
# کل بیماران منحصربه‌فرد
all_patients = df_filtered['subject_id'].unique()
len(all_patients)
# نمایش آمار

1774422

In [24]:
len(df_filtered)

364108227

In [25]:
import pandas as pd
import numpy as np

# امن‌تر: اگر code نال یا غیررشته‌ای بود اذیت نکنه
df['code'] = df['code'].astype('string')
df_filtered = df[~df['code'].str.upper().str.startswith('P/', na=False)].copy()
print("kept rows MDS:", len(df_filtered), " / total:", len(df))


kept rows MDS: 364108227  / total: 448503648


In [26]:
import pandas as pd
import numpy as np

# اطمینان از نوع‌ها (برای خروجی تمیز و بدون خطا)
df_filtered = df_filtered.copy()
df_filtered['subject_id']    = pd.to_numeric(df_filtered['subject_id'], errors='coerce').astype('Int64')
df_filtered['numeric_value'] = pd.to_numeric(df_filtered['numeric_value'], errors='coerce').astype('float32')
df_filtered['time']          = pd.to_datetime(df_filtered['time'], errors='coerce', utc=False)

# ردیف‌های بدون subject_id را حذف کنیم (نمی‌توان شارد کرد)
df_filtered = df_filtered.dropna(subset=['subject_id']).copy()
df_filtered['subject_id'] = df_filtered['subject_id'].astype('int64')

# ۳۶ شارد: هر بیمار فقط در یک فایل (mod 36)
N_SHARDS = 45
df_filtered['__shard__'] = (df_filtered['subject_id'] % N_SHARDS).astype('int16')

print("rows to write:", len(df_filtered))


rows to write: 364108227


In [27]:
import numpy as np
import os

N_SHARDS = 36   #45 for Whole # 36 when we have split
OUT_DIR = "./_TrainMDPS_withoutP_sharded"
os.makedirs(OUT_DIR, exist_ok=True)

cols_out = ['subject_id', 'time', 'code', 'numeric_value']

# فرض: subject_id قبلاً int64 شده و NaNها حذف شده‌اند (طبق سلول قبلی‌ات)
sid_mod = (df_filtered['subject_id'].to_numpy(dtype=np.int64, copy=False) % N_SHARDS)

written = 0
for k in range(N_SHARDS):
    mask = (sid_mod == k)                 # بدون ستون اضافی، فقط یک آرایهٔ NumPy
    part = df_filtered.loc[mask, cols_out].sort_values(['subject_id','time'])
    # اگر می‌خوای حتماً ۳۶ فایل 0..35 داشته باشی حتی اگه خالی باشن:
    # if part.empty:
    #     part = part.iloc[0:0]  # فایل صفر-سطر با همان ستون‌ها
    part.to_parquet(os.path.join(OUT_DIR, f"{k}.parquet"),
                    engine="pyarrow", compression="snappy", index=False)
    written += 1

print(f"Done. wrote {written} parquet files into {OUT_DIR}")


Done. wrote 36 parquet files into ./_TrainMDPS_withoutP_sharded


In [28]:
import pyarrow.parquet as pq

OUT_DIR = "./_TrainMDPS_withoutP_sharded"

def parquet_num_rows(path):
    pf = pq.ParquetFile(path)
    md = pf.metadata
    return sum(md.row_group(i).num_rows for i in range(md.num_row_groups))

total = 0
for f in sorted(p for p in os.listdir(OUT_DIR) if p.endswith('.parquet')):
    n = parquet_num_rows(os.path.join(OUT_DIR, f))
    total += n
    print(f, "rows:", n)
print("TOTAL rows:", total)


0.parquet rows: 10043407
1.parquet rows: 10151372
10.parquet rows: 9793578
11.parquet rows: 10092974
12.parquet rows: 9975623
13.parquet rows: 10179572
14.parquet rows: 10124895
15.parquet rows: 10079970
16.parquet rows: 10011528
17.parquet rows: 9928734
18.parquet rows: 10188270
19.parquet rows: 10397216
2.parquet rows: 10239299
20.parquet rows: 10100321
21.parquet rows: 10209444
22.parquet rows: 10357913
23.parquet rows: 10186352
24.parquet rows: 9982441
25.parquet rows: 10133390
26.parquet rows: 10204998
27.parquet rows: 10039670
28.parquet rows: 10332930
29.parquet rows: 9901901
3.parquet rows: 9991486
30.parquet rows: 9970874
31.parquet rows: 10138176
32.parquet rows: 10255827
33.parquet rows: 10122595
34.parquet rows: 9858738
35.parquet rows: 10111323
4.parquet rows: 9894349
5.parquet rows: 10410174
6.parquet rows: 10216435
7.parquet rows: 9925785
8.parquet rows: 10313493
9.parquet rows: 10243174
TOTAL rows: 364108227


In [29]:
from azureml.data.datapath import DataPath
from azureml.data.dataset_factory import FileDatasetFactory
import os

OUT_DIR = "./_TrainMDPS_withoutP_sharded"
DST_PREFIX = "Zahra/012026/Data/MEDS_MDS/data/train"  #held_out"  # بدون اسلشِ اول
''

# اطمینان: پوشه خروجی وجود دارد و فایل parquet داخلش هست
print("Local files to upload:", len([f for f in os.listdir(OUT_DIR) if f.endswith(".parquet")]))

# مقصد روی Datastore
target = DataPath(datastore, DST_PREFIX)

# آپلود همه محتویات OUT_DIR به DST_PREFIX
_ = FileDatasetFactory.upload_directory(
    src_dir=OUT_DIR,
    target=target,
    overwrite=True,
    show_progress=True,
)
print(f"Uploaded to datastore path: {DST_PREFIX}")


Local files to upload: 36
Validating arguments.
Arguments validated.
'overwrite' is set to True. Any file already present in the target will be overwritten.
Uploading files from '/mnt/batch/tasks/shared/LS_root/mounts/clusters/zahra/code/Users/zahra.sobhaninia/Corebehrt_OOT/corebehrt/Myconfigs/CreateIdenticalData/_TrainMDPS_withoutP_sharded' to 'Zahra/012026/Data/MEDS_MDS/data/train'
Copying 36 files with concurrency set to 6
Copied /mnt/batch/tasks/shared/LS_root/mounts/clusters/zahra/code/Users/zahra.sobhaninia/Corebehrt_OOT/corebehrt/Myconfigs/CreateIdenticalData/_TrainMDPS_withoutP_sharded/12.parquet, file 1 out of 36. Destination path: https://forskerpln0ybkrdls01.dfs.core.windows.net/researcher-data/Zahra/012026/Data/MEDS_MDS/data/train/12.parquet
Copied /mnt/batch/tasks/shared/LS_root/mounts/clusters/zahra/code/Users/zahra.sobhaninia/Corebehrt_OOT/corebehrt/Myconfigs/CreateIdenticalData/_TrainMDPS_withoutP_sharded/10.parquet, file 2 out of 36. Destination path: https://forskerpl

In [30]:
paths = Dataset.File.from_files(path=[(datastore, f"{DST_PREFIX}/*.parquet")]).to_path()
print("Found in datastore:", len(paths))
print(paths[:20])


{'infer_column_types': 'False', 'activity': 'to_path'}
{'infer_column_types': 'False', 'activity': 'to_path', 'activityApp': 'FileDataset'}
Found in datastore: 36
['/0.parquet', '/1.parquet', '/10.parquet', '/11.parquet', '/12.parquet', '/13.parquet', '/14.parquet', '/15.parquet', '/16.parquet', '/17.parquet', '/18.parquet', '/19.parquet', '/2.parquet', '/20.parquet', '/21.parquet', '/22.parquet', '/23.parquet', '/24.parquet', '/25.parquet', '/26.parquet']
